# 🎯 Notebook 07 — Throwing Away the Table
### Policy Gradients and REINFORCE

**Series**: RL Notebook Series · Act III — Policy-Based Methods · Post 7 of 15

---

## Welcome to Act III

Up until now, every algorithm you've built (Value Iteration, Q-Learning, DQN) relied on a simple premise: figure out the expected *Value* of every action, and then just confidently pick the highest one (`argmax`).

But what if the action space isn't just `[Left, Right]`? What if you are steering a car, and the action is a continuous steering angle from `-90.0` to `+90.0`? You can't output Q-values for infinite possible steering angles and compute `argmax()`!

Welcome to **Policy Gradients**. We are going to throw away Q-values entirely. Instead of asking a Neural Network "how good is this state?", we are going to directly ask the Neural Network, **"What is the probability I should take each action?"**

By the end of this notebook, you will understand:
1. **Stochastic Policies** — using a Neural Network to output probabilities instead of values.
2. **The Policy Gradient Theorem** — how to use Calculus to push up the probabilities of good actions.
3. **REINFORCE (1992)** — the grandfather of modern Deep RL algorithms.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. The Policy Network $\pi_\theta(a|s)$

In DQN, the network outputs arbitrary real numbers (Q-values). 

In Policy Gradients, the network outputs **Logits**. We then pass those logits through a **Softmax** function to ensure they sum to exactly `1.0`. These are our action probabilities.

For CartPole, if the state is `s`, the network might output `[0.2, 0.8]`. This means the policy $\pi_\theta$ says there is a 20% chance we should move left, and an 80% chance we should move right.

Because the policy outputs a probability distribution rather than a strict rule, we call it a **Stochastic Policy**.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            # Softmax ensures the outputs are probabilities summing to 1.0
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

    def act(self, state):
        """
        Given a state tensor, this function samples an action from the probability distribution.
        Crucially, it must also return the `log_prob` (the logarithm of the probability of the chosen action), 
        which we need later for the Calculus math.
        """
        probs = self.forward(state)
        
        # ============================================================
        # TODO: Implement action sampling using Categorical.
        # 1. Create a Categorical distribution `m` parameterised by `probs`.
        # 2. Sample an `action` from `m`.
        # 3. Return a tuple of: (the raw python int of the action, the log_prob of the action)
        # ============================================================
        raise NotImplementedError

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

1. `m = Categorical(probs)`
2. `action = m.sample()`
3. `return action.item(), m.log_prob(action)`

</details>

## 2. Knowing what happened: The Return $G_t$

In DQN, we updated the network after every single step using the Bellman Equation (bootstrapping). 

REINFORCE takes a different approach. It is a **Monte Carlo** algorithm. This means it collects an *entire* episode from start to finish. Once the episode ends, it looks backward at every step and calculates the true, exact Discounted Return $G_t$ for that specific step.

If the pole fell down on step 10, then step 9 had a terrible return. Step 1 had a slightly better return. We use the formula from Notebook 02:
$$ G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots $$

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """
    Takes a list of rewards for an entire episode and computes the discounted return G_t for each step.
    """
    returns = []
    R = 0
    
    # ============================================================
    # TODO: Calculate the discounted returns G_t for each step.
    # Iterate backwards through the `rewards` list.
    # Update the discounted sum R.
    # Insert R at the *beginning* of the `returns` list so it stays in the correct order.
    # ============================================================
    raise NotImplementedError
    
    returns = torch.tensor(returns)
    
    # EXPERT TRICK: Stabilizing REINFORCE
    # Raw returns can be highly variable. If we subtract the mean and divide by the standard deviation,
    # we roughly center our returns around 0. This gives the network a nice mix of "positive" updates 
    # and "negative" updates, which dramatically accelerates training.
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    return returns

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

```python
for r in reversed(rewards):
    R = r + gamma * R
    returns.insert(0, R)
```

</details>

## 3. The Math behind the Magic

This is the core of Policy Gradients. 

If we took Action $A$ in State $S$, and eventually achieved a massive Return $G$, we want to *increase* the probability of taking Action $A$ next time we see State $S$.

Without getting bogged down in the massive calculus proof (the Policy Gradient Theorem), the final result of the derivative reveals an incredibly simple loss function for our Neural Network:

$$ \text{Loss} = - \sum_{t=0}^{T} \big( \log \pi_\theta(a_t | s_t) \cdot G_t \big) $$

Let's break this down:
- $\log \pi_\theta(a_t | s_t)$ is just the `log_prob` we returned earlier.
- $G_t$ is the return we just calculated.
- We multiply them together. If $G_t$ is highly positive, the math says "make $\log_\text{prob}$ higher". (We add a negative sign at the front because PyTorch is designed to *minimize* loss, and we want to *maximize* this value).

That's it. That's the entire learning objective.

## 4. The Training Loop: REINFORCE

Because we need the true returns $G_t$, REINFORCE cannot use a Replay Buffer. It must play out an entire episode, update its weights based on that exact episode, throw the data away, and collect a brand new one.

We call algorithms that must learn directly from their own real-time actions **On-Policy** methods (as opposed to DQN, which is **Off-Policy** because it can learn from old replay data).

In [ ]:
def train_reinforce(env_name='CartPole-v1', num_episodes=500, gamma=0.99, lr=1e-2):
    env = gym.make(env_name)
    
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy_net = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    
    episode_rewards = []
    
    for ep in range(num_episodes):
        state, _ = env.reset()
        
        # Trackers for the current episode
        log_probs = []
        rewards = []
        done = False
        truncated = False
        
        # 1. Play out the entire episode
        while not (done or truncated):
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            action, log_prob = policy_net.act(state_tensor)
            
            next_state, reward, done, truncated, _ = env.step(action)
            
            log_probs.append(log_prob)
            rewards.append(reward)
            
            state = next_state
            
        episode_rewards.append(sum(rewards))
        
        # 2. Episode is over! Time to learn.
        returns = compute_returns(rewards, gamma)
        
        # 3. Calculate Policy Loss: - sum(log_prob * G)
        policy_loss = []
        # ============================================================
        # TODO: Calculate the negative REINFORCE Loss.
        # For every `log_prob` and corresponding `G` in `returns`:
        #     append `-log_prob * G` to the `policy_loss` list.
        # ============================================================
        raise NotImplementedError
            
        # Sum up all the log_probs in the list into a single computational graph tensor
        policy_loss = torch.cat(policy_loss).sum()
        
        # 4. Backpropagate Standard PyTorch Steps
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()
        
        if (ep + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            print(f"Episode {ep+1:3d} | Avg Reward: {avg:.1f}")
            
    return policy_net, episode_rewards


<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

```python
for log_prob, G in zip(log_probs, returns):
    policy_loss.append(-log_prob * G)
```

</details>

In [ ]:
trained_net, reinforce_rewards = train_reinforce(num_episodes=500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(reinforce_rewards, color='#ef4444', alpha=0.3)

window = 20
if len(reinforce_rewards) >= window:
    smoothed = np.convolve(reinforce_rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(reinforce_rewards)), smoothed, color='#ef4444', linewidth=2, label=f'REINFORCE (Moving Avg)')

ax.set_title("REINFORCE Training on CartPole-v1")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.axhline(500, color='gray', linestyle='--', label="Max Reward")
ax.legend()
plt.show()

## Watch the Agent Play!

Let's render the environment and see how well our trained agent performs! We'll use `matplotlib.animation` to generate an HTML5 video directly in the notebook.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def render_agent(env_name, agent):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset(seed=42)
    frames = []
    
    done = False
    truncated = False
    while not (done or truncated) and len(frames) < 500:
        frames.append(env.render())
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            action, _ = agent.act(state_tensor)
        state, _, done, truncated, _ = env.step(action)
    env.close()
    
    fig, ax = plt.subplots(figsize=(6,4))
    ax.axis('off')
    img = ax.imshow(frames[0])
    
    def animate(i):
        img.set_data(frames[i])
        return [img]
        
    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    plt.close() # Prevent extra static plot from showing
    return HTML(anim.to_jshtml())

render_agent('CartPole-v1', trained_net)


## Conclusion

REINFORCE successfully learns to solve CartPole! It does so entirely without tables, without overestimation bias, and without moving targets. It directly learned what to do by throwing dice (sampling actions) and evaluating if the eventual outcome was good or bad.

But REINFORCE has a massive weakness: **High Variance**.

Because it waits until the end of the entire episode to figure out if actions were good or bad, the signal is incredibly noisy. If the agent took 100 amazing moves, but then 1 fatally stupid move at the end that crashed the cart... REINFORCE assigns a terrible $G_t$ to *all 101 moves*.

It requires hundreds of episodes for this noise to statistically average out and point the gradient in the correct direction.

### What's Next?
We have Value Methods (DQN), which are great at estimating the exact value of a state instantly, but suffer from bias and instability.
We have Policy Methods (REINFORCE), which guarantee convergence to the optimal policy, but suffer from massive variance and take forever to learn.

In **Notebook 08**, we combine them. We will train a fast, unbiased Policy network (the Actor) and stabilize it using a Value network baseline (the Critic). Welcome to **Actor-Critic** architectures.